# 03 - Desenvolvimento de Sensores
> Circuitos de condicionamento, firmware e calculos de desempenho

**Modulo:** `12_enfitec_servicos` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---


In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
HAIKU  = 'claude-haiku-4-5-20251001'
SONNET = 'claude-sonnet-4-5'

def ask(prompt, system=None, model=HAIKU, max_tokens=1024):
    kw = dict(model=model, max_tokens=max_tokens,
              messages=[{'role':'user','content':prompt}])
    if system: kw['system'] = system
    return client.messages.create(**kw).content[0].text

print('Pronto!')

## Circuito de Condicionamento de Sinal

O condicionamento de sinal e essencial para converter a saida bruta de um sensor
em um sinal adequado para leitura por um ADC. Vamos projetar uma cadeia completa
para um sensor de strain gauge.

In [ ]:
system_eng = (
    "You are an electronics engineer specialized in sensor signal conditioning. "
    "Always show component value calculations with formulas. "
    "Use ASCII-only characters in all responses."
)

resp = ask(
    "Design a complete signal conditioning chain for a 350 ohm strain gauge sensor "
    "(gauge factor 2.1, max strain 2000 microstrain). Include:\n"
    "1. Wheatstone bridge configuration with excitation voltage selection\n"
    "2. Instrumentation amplifier (INA128) with gain calculation\n"
    "3. Anti-aliasing filter design\n"
    "4. ADC interface (ADS1115, 16-bit) with voltage reference\n\n"
    "Show all component value calculations step by step.",
    system=system_eng,
    model=SONNET,
    max_tokens=2048
)
print(resp)

In [ ]:
# Calculos numericos do circuito de condicionamento
R_gauge = 350        # ohms
GF = 2.1             # gauge factor
epsilon_max = 2000e-6 # max strain (2000 microstrain)
V_exc = 5.0          # excitation voltage (V)

# Variacao de resistencia do strain gauge
delta_R = R_gauge * GF * epsilon_max
print(f"Variacao de resistencia: {delta_R:.3f} ohms")

# Tensao de saida da ponte de Wheatstone (desbalanceada)
V_bridge = V_exc * GF * epsilon_max / 4
print(f"Tensao de saida da ponte: {V_bridge*1000:.3f} mV")

# Ganho necessario do amplificador para fundo de escala de 3.3V
V_adc_max = 3.3
gain_needed = V_adc_max / V_bridge
print(f"Ganho necessario: {gain_needed:.1f}")

# Resistor de ganho do INA128: G = 1 + 50k/Rg
Rg = 50000 / (gain_needed - 1)
print(f"Resistor de ganho (Rg): {Rg:.1f} ohms")

## Firmware de Leitura de Sensores

Firmware para microcontrolador que le sensores analogicos, aplica calibracao,
filtra ruido e transmite dados.

In [ ]:
resp = ask(
    "Generate complete Arduino/ESP32 firmware code for a sensor reading system with:\n"
    "1. Analog sensor reading (ADC pin 34) with oversampling (16 samples)\n"
    "2. Calibration curve application (3rd degree polynomial)\n"
    "3. Moving average filter (window size 10)\n"
    "4. Serial output at 115200 baud with timestamp\n"
    "5. WiFi transmission via HTTP POST to a server endpoint\n\n"
    "Include comments explaining each section. Use ESP32 (Arduino framework).",
    system=system_eng,
    model=SONNET,
    max_tokens=2048
)
print(resp)

In [ ]:
# Simulacao do filtro de media movel em Python
import numpy as np

np.random.seed(42)
# Sinal simulado: sensor com ruido
t = np.linspace(0, 10, 500)
sinal_real = 25.0 + 5.0 * np.sin(2 * np.pi * 0.2 * t)
ruido = np.random.normal(0, 0.8, len(t))
sinal_ruidoso = sinal_real + ruido

# Filtro de media movel
def moving_average(data, window=10):
    return np.convolve(data, np.ones(window)/window, mode='valid')

sinal_filtrado = moving_average(sinal_ruidoso, window=10)

print(f"Desvio padrao do ruido original: {np.std(ruido):.4f}")
residuo = sinal_filtrado - sinal_real[9:]
print(f"Desvio padrao apos filtro: {np.std(residuo):.4f}")
print(f"Reducao de ruido: {(1 - np.std(residuo)/np.std(ruido))*100:.1f}%")

## Calculos de Sensibilidade e Resolucao

Especificacoes tecnicas fundamentais para caracterizar um sensor.

In [ ]:
resp = ask(
    "For a custom pressure sensor system with these characteristics:\n"
    "- Pressure range: 0 to 100 bar\n"
    "- Output voltage: 0.5V to 4.5V (ratiometric, 5V supply)\n"
    "- ADC: 12-bit, Vref = 5.0V\n"
    "- Nonlinearity: +/- 0.25% FSO\n\n"
    "Calculate and explain:\n"
    "1. Sensitivity (mV/bar)\n"
    "2. Resolution in bar (1 LSB)\n"
    "3. Total measurement range and span\n"
    "4. Linearity error in bar\n"
    "5. Accuracy budget (combine all error sources)\n\n"
    "Format the results as they would appear in a professional datasheet.",
    system=system_eng,
    model=SONNET,
    max_tokens=2048
)
print(resp)

In [ ]:
# Calculos de especificacao do sensor
P_min, P_max = 0, 100      # bar
V_min, V_max = 0.5, 4.5    # volts
ADC_bits = 12
V_ref = 5.0
nonlinearity_pct = 0.25    # % FSO

# Sensibilidade
sensitivity = (V_max - V_min) / (P_max - P_min)
print(f"Sensibilidade: {sensitivity*1000:.1f} mV/bar")

# Resolucao do ADC
V_lsb = V_ref / (2**ADC_bits)
P_lsb = V_lsb / sensitivity
print(f"Resolucao ADC: {V_lsb*1000:.3f} mV/LSB")
print(f"Resolucao pressao: {P_lsb:.4f} bar/LSB ({P_lsb*1000:.2f} mbar/LSB)")

# Erro de linearidade
lin_error_bar = nonlinearity_pct / 100 * (P_max - P_min)
print(f"Erro de linearidade: +/- {lin_error_bar:.2f} bar")

# Numero efetivo de bits (ENOB) considerando linearidade
import math
enob = math.log2((P_max - P_min) / lin_error_bar)
print(f"ENOB (limitado por linearidade): {enob:.1f} bits")

## Calibracao de Sensores

Rotina de calibracao multi-ponto com ajuste polinomial e analise estatistica.

In [ ]:
resp = ask(
    "Create a complete Python sensor calibration routine that:\n"
    "1. Generates synthetic calibration data for a pressure sensor (0-100 bar)\n"
    "   with realistic nonlinearity and noise\n"
    "2. Implements multi-point calibration (at least 11 points)\n"
    "3. Fits polynomials of degree 1, 2, and 3\n"
    "4. Calculates R-squared for each fit\n"
    "5. Calculates residuals and max error for each\n"
    "6. Recommends the best polynomial degree\n\n"
    "Use numpy and show all code ready to run.",
    system=system_eng,
    model=SONNET,
    max_tokens=2048
)
print(resp)

In [ ]:
import numpy as np

np.random.seed(123)

# Pontos de calibracao (referencia vs leitura do sensor)
p_ref = np.linspace(0, 100, 11)  # pressao de referencia (bar)

# Sensor com nao-linearidade e ruido
v_sensor = (0.5 + 0.04 * p_ref
            + 2e-5 * p_ref**2          # nao-linearidade quadratica
            + np.random.normal(0, 0.01, len(p_ref)))  # ruido

# Ajuste polinomial
for grau in [1, 2, 3]:
    coefs = np.polyfit(v_sensor, p_ref, grau)
    p_fit = np.polyval(coefs, v_sensor)
    
    # R-quadrado
    ss_res = np.sum((p_ref - p_fit)**2)
    ss_tot = np.sum((p_ref - np.mean(p_ref))**2)
    r2 = 1 - ss_res / ss_tot
    
    # Erro maximo
    max_err = np.max(np.abs(p_ref - p_fit))
    
    print(f"Grau {grau}: R2 = {r2:.8f}, Erro max = {max_err:.4f} bar")
    print(f"  Coeficientes: {coefs}")

print("\nRecomendacao: usar grau 2 (corrige a nao-linearidade quadratica)")

## Analise de Ruido e SNR

Identificacao de fontes de ruido, calculo de SNR e tecnicas de filtragem.

In [ ]:
resp = ask(
    "Explain the main noise sources in a sensor measurement system:\n"
    "1. Thermal noise (Johnson-Nyquist)\n"
    "2. Shot noise\n"
    "3. 1/f (flicker) noise\n"
    "4. Quantization noise\n"
    "5. EMI/RFI interference\n\n"
    "For each, give the formula, typical magnitude, and mitigation strategy.\n"
    "Then show how to calculate total SNR for a system with:\n"
    "- Signal amplitude: 2.0V peak\n"
    "- Bandwidth: 1kHz\n"
    "- Source resistance: 10k ohm\n"
    "- Temperature: 25C\n"
    "- ADC: 16-bit, 5V range",
    system=system_eng,
    model=SONNET,
    max_tokens=2048
)
print(resp)

In [ ]:
import numpy as np

# Calculos de ruido
k_B = 1.38e-23       # constante de Boltzmann (J/K)
T = 298.15            # temperatura (K) = 25C
R = 10000             # resistencia (ohms)
BW = 1000             # largura de banda (Hz)
V_signal = 2.0        # amplitude do sinal (V pico)

# Ruido termico (Johnson-Nyquist)
V_thermal = np.sqrt(4 * k_B * T * R * BW)
print(f"Ruido termico: {V_thermal*1e6:.2f} uV rms")

# Ruido de quantizacao (ADC 16-bit, 5V)
ADC_bits = 16
V_ref = 5.0
V_lsb = V_ref / (2**ADC_bits)
V_quant = V_lsb / np.sqrt(12)  # ruido de quantizacao rms
print(f"Ruido de quantizacao: {V_quant*1e6:.2f} uV rms")

# Ruido total (RSS - root sum of squares)
V_noise_total = np.sqrt(V_thermal**2 + V_quant**2)
print(f"Ruido total (RSS): {V_noise_total*1e6:.2f} uV rms")

# SNR
V_signal_rms = V_signal / np.sqrt(2)  # sinal RMS
SNR_linear = V_signal_rms / V_noise_total
SNR_dB = 20 * np.log10(SNR_linear)
print(f"\nSNR: {SNR_dB:.1f} dB ({SNR_linear:.0f}:1)")

# ENOB a partir do SNR
ENOB = (SNR_dB - 1.76) / 6.02
print(f"ENOB: {ENOB:.1f} bits")

In [ ]:
# Demonstracao de filtragem de sinal ruidoso
import numpy as np

np.random.seed(55)
fs = 10000  # frequencia de amostragem (Hz)
t = np.arange(0, 0.1, 1/fs)  # 100ms de dados

# Sinal: 50Hz + ruido branco + interferencia de 60Hz
sinal = 1.0 * np.sin(2 * np.pi * 50 * t)
ruido = 0.3 * np.random.randn(len(t))
interferencia = 0.5 * np.sin(2 * np.pi * 60 * t)
sinal_total = sinal + ruido + interferencia

# Filtro de media movel (simples)
def ma_filter(data, n=20):
    return np.convolve(data, np.ones(n)/n, mode='same')

filtrado = ma_filter(sinal_total, n=20)

# SNR antes e depois
snr_antes = 20 * np.log10(np.std(sinal) / np.std(ruido + interferencia))
snr_depois = 20 * np.log10(
    np.std(filtrado[100:-100]) /
    np.std(filtrado[100:-100] - sinal[100:-100])
)

print(f"SNR antes do filtro:  {snr_antes:.1f} dB")
print(f"SNR depois do filtro: {snr_depois:.1f} dB")
print(f"Melhoria: {snr_depois - snr_antes:.1f} dB")

## Datasheet do Sensor

Geracao de datasheet profissional a partir de especificacoes tecnicas.

In [ ]:
resp = ask(
    "Generate a professional datasheet for a custom pressure sensor with these specs:\n"
    "- Model: ENFI-PS100\n"
    "- Manufacturer: Enfitec Engenharia\n"
    "- Type: Piezoresistive pressure sensor\n"
    "- Range: 0-100 bar\n"
    "- Output: 4-20mA (2-wire)\n"
    "- Supply: 12-36V DC\n"
    "- Accuracy: +/-0.25% FSO (combined)\n"
    "- Operating temp: -20 to +85C\n"
    "- Process connection: 1/4 NPT male\n"
    "- Housing: 316L stainless steel\n"
    "- IP rating: IP67\n"
    "- Certifications: CE, RoHS\n\n"
    "Include these sections:\n"
    "1. General Description\n"
    "2. Key Features\n"
    "3. Electrical Specifications (table)\n"
    "4. Mechanical Specifications (table)\n"
    "5. Environmental Specifications (table)\n"
    "6. Ordering Information\n"
    "7. Pin/Wiring Diagram (text description)\n\n"
    "Format as a professional technical document.",
    system=system_eng,
    model=SONNET,
    max_tokens=2048
)
print(resp)

---

## Exercicios Propostos

1. **Ponte de Wheatstone** - Modifique os calculos do circuito de condicionamento
   para usar um strain gauge de 120 ohm (GF=2.0) e refaca os calculos de ganho.

2. **Filtro digital** - Implemente um filtro Butterworth de 2a ordem em Python e
   compare com o filtro de media movel para o mesmo sinal ruidoso.

3. **Calibracao termica** - Extenda a rotina de calibracao para incluir compensacao
   de temperatura (calibracao em 3 temperaturas diferentes).

4. **Firmware WiFi** - Peca ao Claude para adicionar protocolo MQTT ao firmware
   do ESP32 em vez de HTTP POST.

5. **Analise de ruido** - Calcule o SNR para um sistema com amplificador operacional
   (ruido de tensao: 5 nV/sqrt(Hz), ruido de corrente: 1 pA/sqrt(Hz)).

---

## Proximos Passos

- **Notebook 04** - Design de PCBs: revisao de esquematicos, calculos de trilhas
  e documentacao para fabricacao
- Explorar sensores digitais (I2C/SPI) e seu interfaceamento
- Implementar sistema completo de aquisicao de dados com dashboard web
- Estudar protocolos industriais: Modbus RTU, 4-20mA HART